# Week 7 Lab — Database II: CRUD from Code
**192-201 Advanced Computer Programming with Generative AI** · Student Handbook

---

### What you will do today
Last week you learned **C**reate (`INSERT`) and **R**ead (`SELECT`).
Today you learn **U**pdate and **D**elete, and then you put all four inside a class, so an object can **save itself**.

By the end of this lab you can:
1. Change a row with `UPDATE` and remove a row with `DELETE` — always with `WHERE`.
2. Use `?` placeholders correctly (and remember the comma in `(x,)`).
3. Check your result with `lastrowid`, `rowcount` and `fetchone()`.
4. Build a class with `save()`, `load()`, `all()`, `update()` and `delete()`.
5. Build a small CRUD app of your own (a to-do list).

### How to use this notebook
- Run the cells **from top to bottom** (Shift + Enter).
- `▶ Run` cells are ready. Just run them and read the output.
- `✍️ Your turn` cells have a **TODO**. Write the code, then run the **check** cell.
- The note `← Slide N` tells you which lecture slide explains the idea.

> **AI level today: Level 2.** You may ask AI for help, but you must **check** the code and write it in the **prompt log** (Part 10).

## Part 0 — Setup: a fresh `school.db`   <sub>← Slide 3</sub>

This cell makes a **fresh** database with the three students from Week 6: Mai, Somchai and Ploy.
It also gives us two small helpers:
- `reset_db()` — start again with the same three students (use it any time things get messy).
- `show()` — print the whole `students` table.

▶ **Run it.**

In [ ]:
import sqlite3

DB = "school.db"

def reset_db():
    """Delete the students table and make it again with 3 students."""
    conn = sqlite3.connect(DB)
    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS students")
    cur.execute("""
        CREATE TABLE students (
            id   INTEGER PRIMARY KEY,
            name TEXT,
            age  INTEGER,
            gpa  REAL
        )
    """)
    cur.executemany("INSERT INTO students (name, age, gpa) VALUES (?, ?, ?)",
                    [("Mai", 20, 3.60), ("Somchai", 22, 3.10), ("Ploy", 21, 3.85)])
    conn.commit()
    conn.close()

def show():
    """Print every row of the students table."""
    conn = sqlite3.connect(DB)
    rows = conn.execute("SELECT id, name, age, gpa FROM students").fetchall()
    conn.close()
    print("id | name     | age | gpa")
    print("---+----------+-----+-----")
    for r in rows:
        print(f"{r[0]:<2} | {r[1]:<8} | {r[2]:<3} | {r[3]}")
    print(f"({len(rows)} rows)")

reset_db()
show()

## Part 1 — The CRUD map (no code)   <sub>← Slides 4–5</sub>

✍️ **Your turn.** Double-click this cell and fill in the blanks. This table is the key idea of the week.

| CRUD action | SQL word | Method in our class | Real app example |
|---|---|---|---|
| **C**reate | `INSERT` | `s.save()` | sign up for an account |
| **R**ead | `______` | `Student.load(id)` / `Student.all()` | `______` |
| **U**pdate | `______` | `______` | change your password |
| **D**elete | `______` | `______` | `______` |

## Part 2 — Update a row   <sub>← Slide 6</sub>

`UPDATE` changes a row that is **already there**. It has three parts:

```sql
UPDATE students      -- which table
SET    gpa = ?       -- the new value
WHERE  id = ?        -- which row
```

▶ **Run it.** Ploy (id 3) has a new GPA: 3.90.

In [ ]:
conn = sqlite3.connect(DB)
cur = conn.cursor()

cur.execute("UPDATE students SET gpa = ? WHERE id = ?", (3.90, 3))
conn.commit()                       # save the change

print("Rows changed:", cur.rowcount)
show()

✍️ **Your turn.** Mai (id 1) had her birthday. Change her **age to 21**.
Use `UPDATE ... SET age = ? WHERE id = ?` and do not forget `commit()`.

In [ ]:
# TODO: change Mai's age to 21



show()

In [ ]:
# ✅ Check (just run)
row = cur.execute("SELECT age FROM students WHERE id = 1").fetchone()
print("✅ Correct! Mai is 21." if row[0] == 21 else "❌ Not yet. Mai's age is " + str(row[0]))

## Part 3 — Delete a row   <sub>← Slide 7</sub>

`DELETE` removes the **whole row**. No column list. No `SET`. Just the table and `WHERE`.

```sql
DELETE FROM students
WHERE id = ?
```

▶ **Run it.** Somchai (id 2) left the course. Look closely at `(2,)` — the comma is important (Part 5).

In [ ]:
cur.execute("DELETE FROM students WHERE id = ?", (2,))
conn.commit()

print("Rows deleted:", cur.rowcount)
show()

✍️ **Your turn.** The cell below adds a test student called **"Temp"**. Then **delete** the row where the **name** is `"Temp"`.
Hint: `WHERE name = ?` and the value is `("Temp",)`.

In [ ]:
cur.execute("INSERT INTO students (name, age, gpa) VALUES (?, ?, ?)", ("Temp", 99, 0.0))
conn.commit()
show()

# TODO: delete the row where name is "Temp"



show()

In [ ]:
# ✅ Check (just run)
left = cur.execute("SELECT COUNT(*) FROM students WHERE name = 'Temp'").fetchone()[0]
print("✅ Correct! Temp is gone." if left == 0 else "❌ Temp is still in the table.")

## Part 4 — The WHERE trap (a safe test)   <sub>← Slide 8</sub>

What happens if we **forget `WHERE`**? Let's see — safely.
We will **not** call `commit()`. Instead we call **`rollback()`**, which means *undo everything since the last commit* (Week 6).

▶ **Run it and read the output carefully.**

In [ ]:
print("BEFORE:")
show()

# ⚠️ No WHERE!
cur.execute("UPDATE students SET gpa = 4.0")
print("\nRows changed:", cur.rowcount)

rows = cur.execute("SELECT name, gpa FROM students").fetchall()
print("Inside this connection it looks like:", rows)

conn.rollback()      # UNDO — we did NOT commit
print("\nAFTER rollback (nothing was saved):")
show()

✍️ **Answer in this cell** (double-click to edit):

1. How many rows did `UPDATE students SET gpa = 4.0` change? → `____`
2. Which rows would `DELETE FROM students` remove? → `____`
3. Why is the table normal again after `rollback()`? → `____`
4. What good habit stops this mistake? → `____`

## Part 5 — `?` placeholders and the comma   <sub>← Slide 9</sub>

Rules:
- **One `?` for each value**, in the same order.
- The values go in a **tuple**. One value needs a **comma**: `(2,)`.
- **Never** put values into SQL with an f-string.

▶ **Run it.** This code has the comma mistake. Read the error message.

In [ ]:
try:
    cur.execute("SELECT name FROM students WHERE id = ?", (1))     # ❌ no comma
    print(cur.fetchone())
except Exception as e:
    print("ERROR:", type(e).__name__, "-", e)

print("(1)  is:", type((1)))
print("(1,) is:", type((1,)))

✍️ **Your turn — fix two bugs.**
1. `find_by_id` forgets the comma. Fix it.
2. `find_by_name` uses an f-string. It breaks on a name like `O'Brien`. Rewrite it with `?`.

In [ ]:
def find_by_id(student_id):
    cur.execute("SELECT name FROM students WHERE id = ?", (student_id))     # TODO: fix
    return cur.fetchone()

def find_by_name(name):
    cur.execute(f"SELECT id, name FROM students WHERE name = '{name}'")    # TODO: fix
    return cur.fetchone()

In [ ]:
# ✅ Check (just run)
cur.execute("INSERT INTO students (name, age, gpa) VALUES (?, ?, ?)", ("O'Brien", 23, 3.2))
conn.commit()
ok = 0
try:
    assert find_by_id(1) == ("Mai",); ok += 1; print("✅ find_by_id works")
except Exception as e:
    print("❌ find_by_id:", type(e).__name__, e)
try:
    assert find_by_name("O'Brien")[1] == "O'Brien"; ok += 1; print("✅ find_by_name works with O'Brien")
except Exception as e:
    print("❌ find_by_name:", type(e).__name__, e)
print(f"{ok}/2 fixed")

## Part 6 — Did it work?   <sub>← Slide 10</sub>

| Use | After | It tells you |
|---|---|---|
| `cur.lastrowid` | `INSERT` | the new id |
| `cur.rowcount` | `UPDATE` / `DELETE` | how many rows changed (0 = none found) |
| `cur.fetchone()` | `SELECT` | one row, or `None` |

✍️ **Predict first!** Write your guess for each `print` in a comment, **then** run the cell.

In [ ]:
cur.execute("INSERT INTO students (name, age, gpa) VALUES (?, ?, ?)", ("Nok", 19, 3.4))
conn.commit()
print("A:", cur.lastrowid)          # my guess:

cur.execute("UPDATE students SET gpa = ? WHERE id = ?", (3.5, 99))
conn.commit()
print("B:", cur.rowcount)           # my guess:

cur.execute("SELECT name FROM students WHERE id = ?", (99,))
print("C:", cur.fetchone())         # my guess:

conn.close()     # we are done with this connection

## Part 7 — The object saves itself: `Student` with CRUD   <sub>← Slides 11–16</sub>

Now the big idea: put the SQL **inside the class**. Then the rest of the program just says `s.save()`.

| Method | CRUD | SQL | Works on |
|---|---|---|---|
| `save()` | C | `INSERT` | one object (`self`) |
| `load(id)` | R | `SELECT ... WHERE id = ?` | the class (`@classmethod`) |
| `all()` | R | `SELECT` | the class (`@classmethod`) |
| `update()` | U | `UPDATE ... WHERE id = ?` | one object (`self`) |
| `delete()` | D | `DELETE ... WHERE id = ?` | one object (`self`) |

**New words:**
- `id=None` — a new object has **no id** until it is saved.
- `@classmethod` — a method we call on the **class**: `Student.load(3)`. We need it because we do not have the object yet. `cls` means *this class*.

`save()` and `load()` are done for you as a model. ✍️ **Complete `all()`, `update()` and `delete()`.** Every method follows the same rhythm: **connect → cursor → execute → commit → close**.

In [ ]:
class Student:
    def __init__(self, name, age, gpa, id=None):
        self.id = id            # None until saved
        self.name = name
        self.age = age
        self.gpa = gpa

    def __str__(self):
        return f"#{self.id} {self.name}, age {self.age}, GPA {self.gpa}"

    # C — Create  (model: done for you)
    def save(self):
        conn = sqlite3.connect(DB)
        cur = conn.cursor()
        cur.execute("INSERT INTO students (name, age, gpa) VALUES (?, ?, ?)",
                    (self.name, self.age, self.gpa))
        conn.commit()
        self.id = cur.lastrowid         # remember my new id
        conn.close()

    # R — Read one  (model: done for you)
    @classmethod
    def load(cls, id):
        conn = sqlite3.connect(DB)
        cur = conn.cursor()
        cur.execute("SELECT id, name, age, gpa FROM students WHERE id = ?", (id,))
        row = cur.fetchone()
        conn.close()
        if row is None:
            return None                 # not found
        return cls(row[1], row[2], row[3], id=row[0])

    # R — Read all
    @classmethod
    def all(cls):
        # TODO: SELECT every row, use fetchall(), and return a LIST of Student objects
        pass

    # U — Update
    def update(self):
        # TODO: UPDATE name, age and gpa WHERE id = self.id  (4 values, 4 ?)
        pass

    # D — Delete
    def delete(self):
        # TODO: DELETE the row WHERE id = self.id, then set self.id = None
        pass

In [ ]:
# ✅ Check your Student class (just run)
reset_db()
passed = 0
def check(name, test):
    global passed
    try:
        if test():
            passed += 1; print("✅", name)
        else:
            print("❌", name)
    except Exception as e:
        print("❌", name, "->", type(e).__name__, e)

s = Student("Nok", 19, 3.4); s.save()
check("save() gives the new student id 4",   lambda: s.id == 4)
check("load(3) returns Ploy",                lambda: Student.load(3).name == "Ploy")
check("load(99) returns None",               lambda: Student.load(99) is None)
check("all() returns a list of 4 Students",  lambda: len(Student.all()) == 4 and isinstance(Student.all()[0], Student))
def t_update():
    p = Student.load(3); p.gpa = 3.9; p.update()
    return Student.load(3).gpa == 3.9 and Student.load(1).gpa == 3.6   # only Ploy changed
check("update() changes only Ploy",          t_update)
def t_delete():
    m = Student.load(2); m.delete()
    return Student.load(2) is None and m.id is None and len(Student.all()) == 3
check("delete() removes only Somchai",       t_delete)
print(f"\n{passed}/6 checks passed")

## Part 8 — The full story + check in DB Browser   <sub>← Slide 17</sub>

Look at this program: **not one line of SQL**. The class does it all.

> ⚠️ This cell needs your Part 7 class at **6/6**. If you see `'NoneType' object is not iterable`, your `all()` is not finished yet.

▶ **Run it**, then check with your own eyes:
1. Open **DB Browser for SQLite** → **File → Open Database** → `school.db`.
2. Go to **Browse Data** → table `students`. Press **F5** to refresh.
3. The table must match the output below.

> If Python says `database is locked`: close DB Browser (or click **Write Changes**) and run again.

In [ ]:
reset_db()

s = Student("Nok", 19, 3.4)
s.save()                      # C
print("Saved:", s)

p = Student.load(3)           # R
p.gpa = 3.9
p.update()                    # U

m = Student.load(2)
m.delete()                    # D

print("\nEveryone now:")
for st in Student.all():
    print(" ", st)

import os
print("\nYour database file:", os.path.abspath(DB))

## Part 9 — Your turn: a small CRUD app (to-do list)   <sub>apply everything</sub>

Now build it yourself for a **new** entity: a **Task** in a to-do list.

A task has an **id**, a **title** (text) and **done** (finished or not).
SQLite has no True/False type, so we store `done` as an **INTEGER**: `0` = not done, `1` = done.

We use a **new file**, `todo.db`, so your students table is safe.

✍️ Complete the **TODOs**. Copy the rhythm from your `Student` class — it is almost the same.

In [ ]:
TODO_DB = "todo.db"

def reset_todo():
    conn = sqlite3.connect(TODO_DB)
    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS tasks")
    # TODO 1: CREATE TABLE tasks with columns:
    #         id INTEGER PRIMARY KEY, title TEXT, done INTEGER

    conn.commit()
    conn.close()


class Task:
    def __init__(self, title, done=False, id=None):
        self.id = id
        self.title = title
        self.done = done

    def __str__(self):
        box = "[x]" if self.done else "[ ]"
        return f"{box} #{self.id} {self.title}"

    def save(self):
        # TODO 2: INSERT title and done (use int(self.done) to store 0 or 1)
        #         then set self.id = cur.lastrowid
        pass

    @classmethod
    def load(cls, id):
        # TODO 3: SELECT one task by id. Return None if not found.
        #         Build the object with done=bool(row[2])
        pass

    @classmethod
    def all(cls):
        # TODO 4: SELECT all tasks and return a list of Task objects
        pass

    def update(self):
        # TODO 5: UPDATE title and done WHERE id = self.id
        pass

    def delete(self):
        # TODO 6: DELETE WHERE id = self.id, then set self.id = None
        pass

    def mark_done(self):
        """Already done for you: change the object, then save the change."""
        self.done = True
        self.update()

▶ **Use your app.** This is a small to-do "story". Run it **after** your TODOs are done — if `load()` still returns `None` you will see `'NoneType' object has no attribute 'mark_done'`.

In [ ]:
reset_todo()

for title in ["Finish Assignment 2", "Revise for midterm", "Buy coffee"]:
    Task(title).save()

t = Task.load(1)
t.mark_done()                  # U

Task.load(3).delete()          # D

print("My to-do list:")
for task in Task.all():
    print(" ", task)

In [ ]:
# ✅ Check your Task app (just run)
passed = 0
def check(name, test):
    global passed
    try:
        if test():
            passed += 1; print("✅", name)
        else:
            print("❌", name)
    except Exception as e:
        print("❌", name, "->", type(e).__name__, e)

reset_todo()
a = Task("Read chapter 7"); a.save()
b = Task("Wash dishes");    b.save()
check("1-2. table made, save() gives ids 1 and 2", lambda: (a.id, b.id) == (1, 2))
check("3. load(1) returns the right task",        lambda: Task.load(1).title == "Read chapter 7")
check("3. load(99) returns None",                 lambda: Task.load(1) is not None and Task.load(99) is None)
check("4. all() returns 2 Task objects",          lambda: len(Task.all()) == 2 and isinstance(Task.all()[0], Task))
def t5():
    x = Task.load(2); x.mark_done()
    return Task.load(2).done is True and Task.load(1).done is False
check("5. mark_done() updates only one task",     t5)
def t6():
    y = Task.load(1); y.delete()
    return Task.load(1) is None and y.id is None and len(Task.all()) == 1
check("6. delete() removes only one task",        t6)
print(f"\n{passed}/6 checks passed")

⭐ **Stretch (optional, in VS Code):** make a real menu app. Put your `Task` class in `todo.py` and add a loop:

```
1) Add task   2) List tasks   3) Mark done   4) Delete task   0) Quit
```
Use `input()` to read the choice and call the right method. Check `load()` is not `None` before you call `mark_done()` or `delete()`.

## Part 10 — AI prompt log (Level 2)   <sub>← Slide 18</sub>

If you used AI today, fill in one row for **each** prompt. If you did not use AI, write *"No AI used"*.

Check every AI answer with the 5 questions: **WHERE? · every value a `?` · `(x,)` comma · `commit()` · DB Browser shows the right result?**

| # | My prompt | What AI gave me (short) | How I checked it | Accept / Reject + why |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |

## Part 12 — Midterm practice (no AI!)   <sub>← Slide 19</sub>

The midterm next week covers **Weeks 1–7** and is **Level 0 — no AI**. Answer these by **reading**, not running. Write your answers in this cell.

**Q1.** Match each CRUD action to its SQL word: Create · Read · Update · Delete →  `______`·`______`·`______`·`______`

**Q2.** The table `students` has 5 rows. What happens after this code runs? Why?
```python
cur.execute("DELETE FROM students")
conn.commit()
```

**Q3.** Find the bug and fix it:
```python
cur.execute("SELECT * FROM students WHERE id = ?", (7))
```

**Q4.** What does this print if **no** student has id 50?
```python
cur.execute("UPDATE students SET age = ? WHERE id = ?", (30, 50))
conn.commit()
print(cur.rowcount)
```

**Q5.** Explain in 2–3 sentences: why is `s.save()` better than writing the `INSERT` code every time we need it? Use the word **encapsulation**.

**Q6.** Predict the output:
```python
s = Student("Fah", 20, 3.0)
print(s.id)
s.save()          # the table already has ids 1, 2, 3
print(s.id)
s.delete()
print(s.id)
```

**Q7.** Why is `load()` a `@classmethod`, but `update()` is a normal method?

## ✅ End-of-lab checklist

- [ ] Part 2–3: I can write `UPDATE ... SET ... WHERE` and `DELETE FROM ... WHERE`.
- [ ] Part 4: I saw what happens with **no WHERE**, and I know `rollback()` undoes uncommitted changes.
- [ ] Part 5: `2/2 fixed` — I remember the comma in `(x,)` and never use f-strings in SQL.
- [ ] Part 7: `6/6` for my `Student` class.
- [ ] Part 8: I checked `school.db` in DB Browser.
- [ ] Part 9: `6/6` for my `Task` app.
- [ ] Part 10: prompt log filled (or "No AI used").